In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import editdistance

In [ ]:
BATCH_SIZE = 32
EPOCHS = 20
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [59]:
ARABIC_CHARSET = sorted(set("ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأإةى1234567890-+؟.,:()[]{}!\"'،٪؛"))
char2idx = {c: i+1 for i, c in enumerate(ARABIC_CHARSET)}
idx2char = {i: c for c, i in char2idx.items()}
NUM_CLASSES = len(char2idx) +1# +1 for CTC blank

In [60]:
def encode_text(text): 
    return [char2idx[c] for c in text if c in char2idx]
def decode_text(indices): 
    return ''.join([idx2char.get(i, '') for i in indices])

In [61]:
class CRNNDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        image = Image.open(self.data.iloc[idx, 0]).convert("L")
        label = encode_text(str(self.data.iloc[idx, 1]))
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


In [62]:
def collate_fn(batch):
    images, labels = zip(*batch)
    image_tensors = torch.stack(images)
    lengths = torch.tensor([len(l) for l in labels])
    targets = torch.cat(labels)
    return image_tensors, targets, lengths


In [63]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, 2, 1, 0), nn.ReLU(),
        )
        self.rnn1 = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, num_classes + 1)  # +1 for CTC blank

    def forward(self, x):
        x = self.cnn(x)  # [B, 512, 1, W]
        x = x.squeeze(2).permute(0, 2, 1)  # [B, W, 512]
        x, _ = self.rnn1(x)  # [B, W, 512]
        x, _ = self.rnn2(x)  # [B, W, 512]
        x = self.fc(x)  # [B, W, C]
        return x.permute(1, 0, 2)  # [W, B, C] for CTC

In [64]:
train_csv = r"C:\Users\Raihan\OneDrive\Desktop\DPIIT_HACKATHON\train.csv"
val_csv = r"C:\Users\Raihan\OneDrive\Desktop\DPIIT_HACKATHON\val.csv"
english_checkpoint = r"C:\Users\Raihan\OneDrive\Desktop\DPIIT_HACKATHON\crnn_ctc_checkpoint.pth"

In [75]:
BATCH_SIZE = 8
EPOCHS = 20
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [76]:
model = CRNN(num_classes=66)  # assume English had 67 output classes
checkpoint = torch.load(english_checkpoint, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])

# Replace classifier with Arabic head
model.fc = nn.Linear(512, NUM_CLASSES)
model = model.to(DEVICE)

# --- Data Loaders ---
transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor()
])

train_loader = DataLoader(CRNNDataset(train_csv, transform), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(CRNNDataset(val_csv, transform), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# --- Optimizer & Loss ---
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)


In [ ]:
from tqdm import tqdm
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for images, targets, lengths in tqdm(train_loader):
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        input_lengths = torch.full((logits.size(1),), logits.size(0), dtype=torch.long).to(DEVICE)
        loss = criterion(logits, targets, input_lengths, lengths.to(DEVICE))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # --- Validation ---
    model.eval()
    cer_total, acc_total, count = 0, 0, 0
    with torch.no_grad():
        for images, targets, lengths in val_loader:
            images = images.to(DEVICE)
            logits = model(images)
            preds = logits.softmax(2).argmax(2).cpu().numpy().T
            targets_np = targets.numpy()
            start = 0
            for i, l in enumerate(lengths):
                gt = targets_np[start:start+int(l)]
                start += l
                pred_seq = [p for j, p in enumerate(preds[i]) if (j == 0 or p != preds[i][j-1]) and p != 0]
                cer_total += editdistance.eval(pred_seq, gt)
                acc_total += int(pred_seq == gt.tolist()) 
                count += len(gt)

    avg_loss = total_loss / len(train_loader)
    cer = cer_total / count
    val_acc = acc_total / len(val_loader.dataset)
    print(f"[Epoch {epoch}] Loss: {avg_loss:.4f} | CER: {cer:.4f} | Val_Acc: {val_acc:.4f}")

 52%|█████▏    | 7461/14298 [26:31<25:41,  4.44it/s]  